In [2]:
!pip install groq python-dotenv

Groq API Key Securely

In [3]:
import os
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


Imports + Client Setup

In [7]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL_NAME = "llama-3.1-8b-instant"

Expert Configuration (MODEL_CONFIG)

In [8]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
You are a Senior Technical Support Engineer.
Be precise, structured, and code-focused.
Provide debugging steps and example fixes.
Keep tone professional and concise.
"""
    },
    "billing": {
        "system_prompt": """
You are a Customer Billing Support Specialist.
Be empathetic and polite.
Explain billing issues clearly.
Reference refund policies and next steps.
Avoid technical jargon.
"""
    },
    "general": {
        "system_prompt": """
You are a friendly and helpful assistant.
Respond conversationally.
Keep answers simple and clear.
"""
    }
}

Tool Function (Bonus)

In [9]:
def get_bitcoin_price():
    # Mock implementation
    return "The current price of Bitcoin is $63,250 (mock value)."

Router Function

In [10]:
def route_prompt(user_input):
    # Rule-first routing for tool usage
    if "bitcoin" in user_input.lower():
        return "tool"

    routing_prompt = f"""
Classify this text into one of these categories:
[technical, billing, general]

Return ONLY the category name.

Text:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a strict intent classifier."},
            {"role": "user", "content": routing_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()

    if category not in ["technical", "billing", "general"]:
        category = "general"

    return category

Orchestrator Function

In [11]:
def process_request(user_input):
    category = route_prompt(user_input)
    print(f"\n[Router Decision] → {category}")

    # Tool handling
    if category == "tool":
        return get_bitcoin_price()

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

Interactive Test Loop

In [ ]:
print("MoE Router Ready 🚀")

while True:
    query = input("\nEnter query (or type 'exit'): ")

    if query.lower() == "exit":
        break

    result = process_request(query)

    print("\nExpert Response:\n")
    print(result)

MoE Router Ready 🚀

Enter query (or type 'exit'): My python script throws IndexError on line 5

[Router Decision] → technical

Expert Response:

**Error Analysis**

* Error Type: IndexError
* Error Location: Line 5

To resolve this issue, I will need more information about your script. Please provide the following:

1. Your Python script (at least the relevant parts)
2. The expected output
3. The actual output (including the full error message)

**Common Causes of IndexError**

1. **Out-of-range indexing**: You're trying to access an index that doesn't exist in a list or string.
2. **Missing list or string elements**: The list or string you're trying to access doesn't contain the expected elements.

**Debugging Steps**

1. **Verify the data**: Check that the list or string you're trying to access contains the expected elements.
2. **Check indexing**: Ensure that the index you're using is within the valid range (e.g., 0 to `len(list) - 1` for a list).
3. **Print the data**: Add print st